# Parallel Batch Geospatial Processing
## HPC Job Array Simulation Using Python concurrent.futures

This notebook demonstrates parallel batch processing of multi-tile USGS DEM
datasets — a pattern directly applicable to HPC environments like SLURM job arrays.

**Key concept:** Each tile is processed independently (embarrassingly parallel).
In a SLURM HPC environment, each `process_single_tile()` call maps to one
`#SBATCH --array` job, with shared input on a networked filesystem and
independent output directories per job ID.

In [ ]:
import sys
sys.path.insert(0, '../src')

from pathlib import Path
import time
import matplotlib.pyplot as plt
import rasterio
import numpy as np
from batch_processor import process_single_tile, run_parallel_batch, merge_tiles
from benchmark import run_benchmark

%matplotlib inline

# Inventory the tiles
tiles = list(Path("../data/raw/dem_tiles").glob("*.tif"))
total_size = sum(t.stat().st_size for t in tiles) / 1e6

print(f"Tiles found:  {len(tiles)}")
print(f"Total size:   {total_size:.0f} MB")
for t in tiles:
    print(f"  {t.name} ({t.stat().st_size / 1e6:.0f} MB)")

In [ ]:
# Run benchmark and display chart
fig = run_benchmark(
    input_dir="../data/raw/dem_tiles",
    output_dir="../data/processed/benchmark",
    max_workers=4
)
plt.show()

In [ ]:
# Display the merged mosaic
with rasterio.open("../data/processed/colorado_dem_mosaic.tif") as src:
    data = src.read(1).astype(float)
    data[data == src.nodata] = np.nan
    extent = [src.bounds.left, src.bounds.right, src.bounds.bottom, src.bounds.top]

fig, ax = plt.subplots(figsize=(10, 8))
im = ax.imshow(data, cmap="terrain", extent=extent, origin="upper")
plt.colorbar(im, ax=ax, label="Elevation (m)")
ax.set_title("Colorado DEM Mosaic — Merged from Parallel Processing", fontsize=13)
ax.set_xlabel("Easting (m)")
ax.set_ylabel("Northing (m)")
plt.tight_layout()
plt.show()

## Connection to HPC Environments

This pipeline uses `ProcessPoolExecutor` to simulate the embarrassingly
parallel pattern common in HPC job arrays:

| Local (this notebook) | HPC SLURM equivalent |
|----------------------|----------------------|
| `ProcessPoolExecutor(max_workers=4)` | `#SBATCH --array=1-N` |
| `process_single_tile(tile_path)` | One array task per tile |
| Shared `data/raw/` folder | Networked filesystem (Lustre/GPFS) |
| `merge_tiles()` | Post-processing job with `--dependency=afterok:$ARRAY_JOB_ID` |

On NLR's HPC system, the same logic scales from 4 local cores
to hundreds of compute nodes with minimal code changes.